In [ ]:
# Run in a notebook cell (prefix with !)
!pip install -q transformers datasets evaluate accelerate scikit-learn sentencepiece


In [ ]:
!pip install -U transformers


In [ ]:
from datasets import load_dataset, concatenate_datasets
import numpy as np
import pandas as pd

ds = load_dataset("unswnlporg/BESSTIE")
print(ds)   # shows splits

# Inspect each split separately
for split_name, split in ds.items():
    labels = np.array(split['label'])
    unique, counts = np.unique(labels, return_counts=True)
    print(f"Split: {split_name} — rows: {len(split)}")
    print("  Unique label values:", unique)
    print("  Counts per label:", dict(zip(unique, counts)))
    print()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'variety', 'source', 'task'],
        num_rows: 17760
    })
    validation: Dataset({
        features: ['text', 'label', 'variety', 'source', 'task'],
        num_rows: 2428
    })
})
Split: train — rows: 17760
  Unique label values: [0 1]
  Counts per label: {np.int64(0): np.int64(12092), np.int64(1): np.int64(5668)}

Split: validation — rows: 2428
  Unique label values: [0 1]
  Counts per label: {np.int64(0): np.int64(1653), np.int64(1): np.int64(775)}



In [ ]:
from datasets import load_dataset
ds = load_dataset("unswnlporg/BESSTIE")
print(ds)                # shows splits and sizes
# preview features and a few examples
for split in ds:
    print("====", split, "====")
    print(ds[split].column_names)
    print(ds[split][0])
    break


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'variety', 'source', 'task'],
        num_rows: 17760
    })
    validation: Dataset({
        features: ['text', 'label', 'variety', 'source', 'task'],
        num_rows: 2428
    })
})
==== train ====
['text', 'label', 'variety', 'source', 'task']
{'text': "This was one of the best dishes I've EVER had! I have very high standards, and I also comparing to Melbourne cafes. There was a generous amount of mushrooms, perfectly cooked. There was an AMAZING onion jam thing, the truffle mayo was absolutely incredible! Very reasonably priced. I asked for sourdough instead of brioche which was happily done, but the bread was very soggy when I got the plate. I could move to Mackay just for this And as vegetarians, I REALLY appreciate you for not charging us the extra 4 for replacing the bacon with avo in your ' bacon and eggs '. 80 % of places make you pay for substituting the meat for vegetables:", 'label': 1, 'variety': 'en

In [ ]:
from datasets import load_dataset

ds = load_dataset("unswnlporg/BESSTIE")

train_ds = ds["train"]        # training split
eval_ds  = ds["validation"]   # evaluation (test) split

print(train_ds)
print(eval_ds)


Dataset({
    features: ['text', 'label', 'variety', 'source', 'task'],
    num_rows: 17760
})
Dataset({
    features: ['text', 'label', 'variety', 'source', 'task'],
    num_rows: 2428
})


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_tok = train_ds.map(tokenize, batched=True)
eval_tok  = eval_ds.map(tokenize, batched=True)

train_tok.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)
eval_tok.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)


Map:   0%|          | 0/2428 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import f1_score

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1": f1_score(labels, preds)}

# Training arguments (NO evaluation_strategy)
training_args = TrainingArguments(
    output_dir="/mnt/data/finetuned_roberta",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    compute_metrics=compute_metrics
)

trainer.train()



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.631733
200,0.618270
300,0.600135
400,0.630472
500,0.573345
600,0.597639
700,0.582582
800,0.575514
900,0.618012
1000,0.606314


TrainOutput(global_step=4440, training_loss=0.5813086741679424, metrics={'train_runtime': 1529.5585, 'train_samples_per_second': 23.222, 'train_steps_per_second': 2.903, 'total_flos': 4672852343193600.0, 'train_loss': 0.5813086741679424, 'epoch': 2.0})

In [ ]:
import numpy as np

predictions = trainer.predict(eval_tok)
finetuned_preds = np.argmax(predictions.predictions, axis=1)


In [ ]:
len(finetuned_preds), len(eval_ds)


(2428, 2428)

In [ ]:
from sklearn.metrics import classification_report

print("=== Fine-tuned RoBERTa ===")
print(classification_report(eval_ds["label"], finetuned_preds))


=== Fine-tuned RoBERTa ===
              precision    recall  f1-score   support

           0       0.76      0.74      0.75      1653
           1       0.48      0.51      0.49       775

    accuracy                           0.66      2428
   macro avg       0.62      0.62      0.62      2428
weighted avg       0.67      0.66      0.67      2428



In [ ]:
from transformers import pipeline
import torch

prompt_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

candidate_labels = ["negative", "positive"]


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
prompt_preds = []

for text in eval_ds["text"]:
    out = prompt_model(text, candidate_labels=candidate_labels)
    label = out["labels"][0]
    prompt_preds.append(candidate_labels.index(label))


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
print("=== Prompt-based (zero-shot) ===")
print(classification_report(eval_ds["label"], prompt_preds))


=== Prompt-based (zero-shot) ===
              precision    recall  f1-score   support

           0       0.81      0.62      0.70      1653
           1       0.46      0.69      0.55       775

    accuracy                           0.64      2428
   macro avg       0.64      0.66      0.63      2428
weighted avg       0.70      0.64      0.65      2428



In [ ]:
import pandas as pd

eval_df = pd.DataFrame(eval_ds)
eval_df["finetuned_pred"] = finetuned_preds
eval_df["prompt_pred"] = prompt_preds

for v in ["en-AU", "en-IN", "en-UK"]:
    sub = eval_df[eval_df["variety"] == v]
    print(f"\n🌍 Variety: {v}")

    print("Fine-tuned:")
    print(classification_report(sub["label"], sub["finetuned_pred"]))

    print("Prompt-based:")
    print(classification_report(sub["label"], sub["prompt_pred"]))



🌍 Variety: en-AU
Fine-tuned:
              precision    recall  f1-score   support

           0       0.66      0.77      0.71       459
           1       0.50      0.37      0.42       283

    accuracy                           0.62       742
   macro avg       0.58      0.57      0.57       742
weighted avg       0.60      0.62      0.60       742

Prompt-based:
              precision    recall  f1-score   support

           0       0.70      0.67      0.69       459
           1       0.50      0.53      0.51       283

    accuracy                           0.62       742
   macro avg       0.60      0.60      0.60       742
weighted avg       0.62      0.62      0.62       742


🌍 Variety: en-IN
Fine-tuned:
              precision    recall  f1-score   support

           0       0.79      0.77      0.78       651
           1       0.45      0.47      0.46       259

    accuracy                           0.69       910
   macro avg       0.62      0.62      0.62       910


In [ ]:
FINE_TUNED_MODELS = [
    "bert-base-uncased",
    "roberta-base",
    "albert-base-v2",
    "microsoft/deberta-v3-base"
]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import f1_score
from datasets import Dataset
import pandas as pd

results = []

for MODEL_NAME in FINE_TUNED_MODELS:
    print(f"\n=== Fine-tuning {MODEL_NAME} ===")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=256
        )

    train_tok = train_ds.map(tokenize, batched=True)
    eval_tok  = eval_ds.map(tokenize, batched=True)

    train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    eval_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {"f1": f1_score(labels, preds)}

    training_args = TrainingArguments(
        output_dir=f"/mnt/data/{MODEL_NAME.replace('/','_')}",
        per_device_train_batch_size=8,
        num_train_epochs=2,
        learning_rate=2e-5,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        compute_metrics=compute_metrics
    )

    trainer.train()

    preds = np.argmax(trainer.predict(eval_tok).predictions, axis=1)

    eval_df = pd.DataFrame(eval_ds)
    eval_df["pred"] = preds

    for v in ["en-AU", "en-IN", "en-UK"]:
        sub = eval_df[eval_df["variety"] == v]
        f1 = f1_score(sub["label"], sub["pred"], average="macro")
        results.append({
            "model": MODEL_NAME,
            "type": "fine-tuned",
            "variety": v,
            "macro_f1": f1
        })



=== Fine-tuning bert-base-uncased ===


Map:   0%|          | 0/2428 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.616283
1000,0.592232
1500,0.593143
2000,0.590263
2500,0.564801
3000,0.558564
3500,0.553602
4000,0.548749


...


=== Fine-tuning microsoft/deberta-v3-base ===


Map:   0%|          | 0/2428 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight      

Step,Training Loss
500,7.882250
1000,0.000000
1500,0.000000
2000,0.000000
2500,0.000000
3000,0.000000
3500,0.000000
4000,0.000000


In [ ]:
results_df = pd.DataFrame(results)
results_df


,model,type,variety,macro_f1
0,bert-base-uncased,fine-tuned,en-AU,0.551633
1,bert-base-uncased,fine-tuned,en-IN,0.559646
2,bert-base-uncased,fine-tuned,en-UK,0.630159
3,roberta-base,fine-tuned,en-AU,0.580728
4,roberta-base,fine-tuned,en-IN,0.619165
5,roberta-base,fine-tuned,en-UK,0.640697
6,albert-base-v2,fine-tuned,en-AU,0.569127
7,albert-base-v2,fine-tuned,en-IN,0.610124
8,albert-base-v2,fine-tuned,en-UK,0.646460
9,microsoft/deberta-v3-base,fine-tuned,en-AU,0.382182


In [ ]:
results_df = pd.DataFrame(results)
results_df


NameError: name 'results' is not defined

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import f1_score
import pandas as pd
import torch

# Load dataset
ds = load_dataset("unswnlporg/BESSTIE")
eval_ds = ds["validation"]
eval_df = pd.DataFrame(eval_ds)

# Load prompt-based model
prompt_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

candidate_labels = ["negative", "positive"]

# Run inference (batched for speed)
prompt_preds = []
BATCH_SIZE = 32
texts = eval_df["text"].tolist()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    outputs = prompt_model(batch, candidate_labels=candidate_labels)
    for out in outputs:
        prompt_preds.append(candidate_labels.index(out["labels"][0]))

eval_df["prompt_pred"] = prompt_preds


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/17760 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2428 [00:00<?, ? examples/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


KeyboardInterrupt: 

In [ ]:
MODEL_NAME = "bert-base-uncased"   # change per model
NUM_LABELS = 2  # or whatever your task uses

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


NameError: name 'compute_metrics' is not defined